In [110]:

import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
from sklearn import metrics
from data.get_data import get_dataframe
from data_processor.calculate_stats import calculate_statistics
from data_processor.data_categorising import categories_columns
from data_processor.data_cleaner import clean_data
from data_processor.fight_stats import finalProcessingForFighter, calculateAverages
from data_processor.data_types_fixes import check_and_process_data_type, drop_col_for_training

In [72]:
og_df = get_dataframe('original.csv')

In [73]:
cleaned_df = clean_data(og_df)

In [74]:
stats_df = calculate_statistics(cleaned_df)

In [75]:
processed_df = finalProcessingForFighter(stats_df)

In [76]:
processed_df = check_and_process_data_type(processed_df)

In [77]:
avg_df = calculateAverages(processed_df)

In [78]:
cat_df = categories_columns(avg_df)

In [79]:
df_for_training = drop_col_for_training(cat_df)

In [80]:
df_for_training = df_for_training.sort_index()
df_for_training

,Total_KD,Total_STR,Total_TD,Total_SUB,Round,Time,Target,Avg_Round_Time,Avg_Round,Fighter_code,Opp_code,Weight_Class_code
0,0,0,0,0,1.0,20.0,0,0.000000,0.000000,1786,2058,8
1,0,0,0,0,1.0,77.0,0,0.000000,0.000000,2009,1794,8
2,0,0,0,0,1.0,91.0,0,0.000000,0.000000,869,1886,8
3,0,0,0,0,1.0,67.0,0,0.000000,0.000000,1329,1160,8
4,0,0,0,0,1.0,67.0,0,0.000000,0.000000,1322,1008,8
...,...,...,...,...,...,...,...,...,...,...,...,...
14819,0,0,0,0,2.0,257.0,0,309.666667,2.333333,914,1943,5
14820,1,289,2,0,2.0,71.0,0,445.875000,3.750000,2358,994,10
14821,0,226,6,2,3.0,300.0,0,300.000000,3.000000,991,2299,13
14822,1,546,14,11,3.0,75.0,0,295.250000,3.250000,1249,1631,11


In [81]:
X = df_for_training.drop("Target", axis=1)
y = df_for_training["Target"]

In [82]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

In [130]:
model = XGBClassifier(
    n_estimators=10000,
    learning_rate=0.01,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=100
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

y_preds = model.predict(X_test)
y_probs = model.predict_proba(X_test)[:, 1]

[0]	validation_0-logloss:0.69163


[50]	validation_0-logloss:0.65116
[100]	validation_0-logloss:0.63076
[150]	validation_0-logloss:0.61747
[200]	validation_0-logloss:0.60905
[250]	validation_0-logloss:0.60362
[300]	validation_0-logloss:0.59865
[350]	validation_0-logloss:0.59561
[400]	validation_0-logloss:0.59305
[450]	validation_0-logloss:0.59017
[500]	validation_0-logloss:0.58723
[550]	validation_0-logloss:0.58472
[600]	validation_0-logloss:0.58281
[650]	validation_0-logloss:0.58079
[700]	validation_0-logloss:0.57885
[750]	validation_0-logloss:0.57728
[800]	validation_0-logloss:0.57533
[850]	validation_0-logloss:0.57421
[900]	validation_0-logloss:0.57270
[950]	validation_0-logloss:0.57144
[1000]	validation_0-logloss:0.57031
[1050]	validation_0-logloss:0.56938
[1100]	validation_0-logloss:0.56849
[1150]	validation_0-logloss:0.56760
[1200]	validation_0-logloss:0.56677
[1250]	validation_0-logloss:0.56592
[1300]	validation_0-logloss:0.56507
[1350]	validation_0-logloss:0.56438
[1400]	validation_0-logloss:0.56367
[1450]	valid

In [131]:
acc = metrics.accuracy_score(y_test, y_preds)
prec = metrics.precision_score(y_test, y_preds)
roc_acc = float(metrics.roc_auc_score(y_test, y_probs))

roc_acc, acc, prec

(0.7609053455132315, 0.6950338600451468, 0.652952380952381)